# 01 · Environmental Claim Detector (ClimateBERT) — train, evaluate, publish

GreenTruth's text side: fine-tune a small climate transformer to tell an
environmental **claim** from a non-claim, then publish it to the Hugging Face Hub.

**Data source**
```
Dataset:   climatebert/environmental_claims  (2,647 expert-annotated sentences)
Paper:     Stammbach et al., ACL 2023 — https://arxiv.org/abs/2209.00507
Dataset:   https://huggingface.co/datasets/climatebert/environmental_claims
Base:      https://huggingface.co/climatebert/distilroberta-base-climate-f
License:   CC BY-NC-SA 4.0 (non-commercial — the trained model inherits it)
Publish to: https://huggingface.co/Rahilgh/greentruth-claim-detector
```

Real dataset, real metrics from your run. The paper reports macro-F1 ≈ 85%; expect
that range, don't assert it. Set the runtime to **GPU**.

| Notebook card | |
|---|---|
| **Type** | Train + publish a model |
| **Purpose** | Fine-tune the ClimateBERT-based environmental-claim detector and publish it to the Hugging Face Hub. |
| **Inputs** | `climatebert/environmental_claims` (downloaded from the Hub); base model `climatebert/distilroberta-base-climate-f`. |
| **Outputs** | Model `Rahilgh/greentruth-claim-detector` on the Hugging Face Hub (public). |
| **Where it runs** | Google Colab with a GPU runtime; publishing needs a Hugging Face **write** token entered at run time (never saved in a cell). |
| **Execution record** | Executed in Colab — record kept in `notebooks/executed/01_claim_detection_training.ipynb`. Do not re-run the publish step without deciding to overwrite the published model. |


### 1 · Install and import

In [ ]:
!pip install -q transformers datasets scikit-learn huggingface_hub matplotlib

In [ ]:
import numpy as np, torch, transformers
from datasets import load_dataset, DatasetDict
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from sklearn.metrics import (precision_recall_fscore_support, accuracy_score,
                             confusion_matrix, ConfusionMatrixDisplay)
import matplotlib.pyplot as plt
print("transformers", transformers.__version__, "| GPU:", torch.cuda.is_available())

### 2 · Load the dataset (text = sentence, label 1 = environmental claim)

In [ ]:
ds = load_dataset("climatebert/environmental_claims")
print(ds)
if "test" not in ds:
    t = ds["train"].train_test_split(test_size=0.2, seed=42)
    ds = DatasetDict({"train": t["train"], "test": t["test"]})
if "validation" not in ds:
    t = ds["train"].train_test_split(test_size=0.1, seed=42)
    ds = DatasetDict({"train": t["train"], "validation": t["test"], "test": ds["test"]})
for s in ds:
    y = ds[s]["label"]; pos = sum(y)
    print(f"{s:12s} n={len(y):5d} claims={pos:4d} non={len(y)-pos:4d}")

### 3 · Tokenize

In [ ]:
MODEL_NAME = "climatebert/distilroberta-base-climate-f"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def prep(b): return tokenizer(b["text"], truncation=True, max_length=256)
enc = ds.map(prep, batched=True)
collator = DataCollatorWithPadding(tokenizer)

### 4 · Model, metrics, training arguments

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2,
    id2label={0: "not_claim", 1: "environmental_claim"},
    label2id={"not_claim": 0, "environmental_claim": 1})

def compute_metrics(p):
    preds = np.argmax(p.predictions if hasattr(p, 'predictions') else p[0], axis=-1)
    labels = p.label_ids if hasattr(p, 'label_ids') else p[1]
    pr, rc, f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    return {"accuracy": accuracy_score(labels, preds), "precision": pr, "recall": rc, "f1": f1}

OUTPUT_DIR = "greentruth-claim-detector"
def make_args(**kw):
    try:    return TrainingArguments(eval_strategy="epoch", save_strategy="epoch", **kw)
    except TypeError:
        return TrainingArguments(evaluation_strategy="epoch", save_strategy="epoch", **kw)
args = make_args(output_dir=OUTPUT_DIR, learning_rate=2e-5,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    num_train_epochs=3, weight_decay=0.01, load_best_model_at_end=True,
    metric_for_best_model="f1", logging_steps=20, report_to="none")

### 5 · Train

In [ ]:
trainer = Trainer(model=model, args=args,
    train_dataset=enc["train"], eval_dataset=enc["validation"],
    tokenizer=tokenizer, data_collator=collator, compute_metrics=compute_metrics)
trainer.train()

### 6 · Evaluate on the held-out test split

In [ ]:
m = trainer.evaluate(enc["test"])
print({k: round(v, 4) for k, v in m.items() if k.startswith("eval_")})
pred = trainer.predict(enc["test"])
y_true, y_pred = pred.label_ids, np.argmax(pred.predictions, axis=-1)
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred),
    display_labels=["not_claim", "claim"]).plot(cmap="Greens")
plt.title("Environmental claim detection — test set"); plt.show()

These are the metrics **your run** produced. The source paper reports macro-F1 ≈ 85% for comparable models; a similar range is expected. Don't edit the numbers.

### 7 · Save locally

In [ ]:
trainer.save_model(OUTPUT_DIR); tokenizer.save_pretrained(OUTPUT_DIR)
print("saved to", OUTPUT_DIR)

### 8 · Publish to the Hugging Face Hub

Log in with a **write** token from https://huggingface.co/settings/tokens. Never
paste a token into a saved cell. Your account is `Rahilgh`.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
HF_USERNAME = "Rahilgh"
REPO_ID = f"{HF_USERNAME}/greentruth-claim-detector"

MODEL_CARD = f"""---
license: cc-by-nc-sa-4.0
language: en
base_model: climatebert/distilroberta-base-climate-f
datasets: [climatebert/environmental_claims]
tags: [text-classification, climate, esg, greenwashing]
---

# GreenTruth Claim Detector

Binary classifier flagging whether a sentence is an environmental claim. Text stage
of GreenTruth, which checks environmental claims against Earth-observation evidence.

## Intended use
Find checkable environmental claims in corporate disclosures, before evidence-grounded
verification. Detecting a claim is not judging it.

## Training
Fine-tuned climatebert/distilroberta-base-climate-f on the environmental_claims dataset
(Stammbach et al., ACL 2023): 3 epochs, lr 2e-5, batch 16, max length 256.

## Evaluation
See the notebook run for test precision/recall/macro-F1/accuracy. The paper reports
macro-F1 around 85% for comparable models.

## Limitations & ethics
English, listed-company text; domain shift is likely. Detects a claim, not its truth.
Inherits CC BY-NC-SA 4.0 (non-commercial). Do not use to label a company deceptive.

## Dataset
https://huggingface.co/datasets/climatebert/environmental_claims
"""
with open(f"{OUTPUT_DIR}/README.md", "w") as f:
    f.write(MODEL_CARD)

model.push_to_hub(REPO_ID); tokenizer.push_to_hub(REPO_ID)
from huggingface_hub import upload_file
upload_file(path_or_fileobj=f"{OUTPUT_DIR}/README.md", path_in_repo="README.md", repo_id=REPO_ID)
print("pushed to https://huggingface.co/" + REPO_ID)

### 9 · Reload from the Hub and run inference

In [ ]:
from transformers import pipeline
clf = pipeline("text-classification", model=REPO_ID)   # or OUTPUT_DIR for the local copy
for s in ["We reduced routine flaring by 30% by 2023 from 2019 levels.",
          "The board met four times during the reporting period.",
          "We are committed to reaching net zero by 2050.",
          "Our teams volunteered thousands of hours in the community."]:
    r = clf(s)[0]; print(f"[{r['label']:18s} {r['score']:.2f}]  {s}")

### 10 · How GreenTruth uses this model

The app calls this detector in place of the rule baseline, keeping the rule detector as an offline fallback. Detection is stage one; the evidence stages decide whether a claim holds up.